# Data wrangling — Base 14: Empresa de Mineração

Este notebook aplica à base `base_14_empresa_mineracao_original.csv` o mesmo fluxo de limpeza visto em aula (notebook *Limpeza de Dados Exemplo Base*): auditoria, padronização estrutural, remoção de duplicidades, conversão de tipos, validação de domínio, normalização categórica, engenharia de atributos temporais, separação treino/teste, clipping por z-score, imputação, one-hot encoding e padronização.

São produzidas duas representações:

- **base limpa** (`base_14_empresa_mineracao_limpa.csv`): valores legíveis, tipos corretos, categorias padronizadas e ausências explícitas — adequada para análise exploratória e dashboard;
- **matrizes de ML** (`dados_prontos_ml_treino.csv` / `dados_prontos_ml_teste.csv`): somente números, sem ausências, prontas para algoritmos.

A variável-alvo adotada para o problema supervisionado é `operacao_com_restricao` (1 quando `status_operacao` for *Operação com restrição* ou *Operação crítica*; 0 quando for *Operação normal*).

In [1]:
import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(objeto):
        print(objeto)
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda valor: f"{valor:,.3f}")

SEMENTE = 42
# Último dia do período coberto pela base (jan/2025 a ago/2026).
DATA_REFERENCIA = pd.Timestamp("2026-08-31")
ARQUIVO_ORIGINAL = Path("base_14_empresa_mineracao_original.csv")

## 1. Leitura da base bruta

A leitura é feita com todas as colunas como texto (`dtype=str`). Assim nenhuma conversão automática esconde problemas: números com `R$`, datas em formatos diferentes e campos vazios ficam visíveis na auditoria.

In [2]:
df_bruto = pd.read_csv(ARQUIVO_ORIGINAL, sep=";", dtype=str, encoding="utf-8-sig")
print(f"Dimensão: {df_bruto.shape}")
display(df_bruto.head())

Dimensão: (10000, 36)


,registro_id,data_operacao,mina,estado,frente_lavra,turno,tipo_minerio,equipamento_escavacao,equipamento_transporte,operador_id,teor_fe_pct,teor_si_pct,umidade_minerio_pct,granulometria_media_mm,massa_extraida_t,massa_processada_t,massa_produto_t,recuperacao_metalurgica_pct,rejeito_gerado_t,distancia_transporte_km,tempo_ciclo_min,qtd_viagens,consumo_diesel_l,consumo_diesel_estimado_l,consumo_energia_kwh,horas_operacao,horas_parada,qtd_falhas_equipamento,horas_desde_manutencao,custo_manutencao_rs,custo_operacional_rs,custo_operacional_estimado_rs,produtividade_t_h,emissao_co2_kg,indice_qualidade_minerio,status_operacao
0,REG-0000001,16/02/2026,Mina Brucutu,MG,F02,Manhã,Minério misto,ESC-05,CAM-102,OP-1713,51.9,6.64,12.58,24.1,1544.55,1513.06,1271.05,84.01,242.01,4.83,34.3,7,90.64,85.34,340.81,6.39,2.49,1,51.4,12703.4,20076.99,19935.59,174.02,243.88,76.4,Operação normal
1,REG-0000002,06/02/2026,Mina Catalão,GO,F01,Manhã,Hematita,ESC-02,CAM-103,OP-1425,67.09,1,6.88,43.4,1517.53,1460.85,1338.77,91.64,122.08,7.07,34.0,6,84.48,84.98,208.41,9.11,1.34,0,114.0,2252.47,8828.68,9233.37,145.27,227.44,100,Operação normal
2,REG-0000003,06/06/2026,Mina Catalão,GO,F06,Tarde,Magnetita,ESC-04,CAM-102,OP-2410,55.99,8.3,4.96,23.1,1627.57,1554.4,1388.76,89.34,165.64,3.6,30.7,8,68.97,64.57,477.48,9.88,3.35,1,26.2,22251.16,33266.53,32896.49,122.97,187.48,86.6,Operação normal
3,REG-0000004,09/04/2026,Mina Catalão,GO,F04,Manhã,Itabirito,ESC-05,CAM-106,OP-1754,55.57,15.44,8.12,61.4,2680.67,2526.33,2013.64,79.71,512.7,4.92,39.0,15,141.94,130.19,555.29,13.74,1.5,1,123.0,22495.55,33019.78,32892.62,175.87,382.51,78.5,Operação normal
4,REG-0000005,07/02/2025,Mina Corumbá,MS,F03,Manhã,Itabirito,ESC-05,CAM-105,OP-1726,46.91,18.45,4.06,59.8,1596.04,1540.21,1285.54,83.47,254.67,5.42,37.0,7,74.81,74.89,497.12,7.0,2.14,1,91.8,23488.64,29482.43,29417.87,174.66,199.51,62.3,Operação normal


## 2. Auditoria inicial da qualidade

In [3]:
def auditar_dataframe(df):
    return pd.DataFrame({
        "tipo_pandas": df.dtypes.astype(str),
        "ausentes": df.isna().sum(),
        "ausentes_pct": (df.isna().mean() * 100).round(2),
        "valores_unicos": df.nunique(dropna=True),
    })


print(f"Duplicidades exatas: {df_bruto.duplicated().sum()}")
print(f"IDs repetidos: {df_bruto['registro_id'].duplicated().sum()}")
display(auditar_dataframe(df_bruto))

print("Valores presentes em colunas categóricas:")
for coluna in ["mina", "estado", "frente_lavra", "turno", "tipo_minerio",
               "equipamento_escavacao", "equipamento_transporte", "status_operacao"]:
    print(f"{coluna}: {df_bruto[coluna].dropna().unique()}")

Duplicidades exatas: 150
IDs repetidos: 150


,tipo_pandas,ausentes,ausentes_pct,valores_unicos
registro_id,str,0,0.000,9850
data_operacao,str,0,0.000,654
mina,str,0,0.000,6
estado,str,0,0.000,4
frente_lavra,str,0,0.000,6
turno,str,0,0.000,12
tipo_minerio,str,119,1.190,16
equipamento_escavacao,str,104,1.040,5
equipamento_transporte,str,111,1.110,8
operador_id,str,0,0.000,1497


Valores presentes em colunas categóricas:
mina: <StringArray>
['Mina Brucutu', 'Mina Catalão', 'Mina Corumbá', 'Mina Azul', 'Mina Itabira', 'Mina Serra Norte']
Length: 6, dtype: str
estado: <StringArray>
['MG', 'GO', 'MS', 'PA']
Length: 4, dtype: str
frente_lavra: <StringArray>
['F02', 'F01', 'F06', 'F04', 'F03', 'F05']
Length: 6, dtype: str
turno: <StringArray>
['Manhã', 'Tarde', 'Noite', 'MANHA', 'manha', 'noite', 'Noite ', 'TARDE', 'Tarde ', 'Manha ', 'tarde', 'NOITE']
Length: 12, dtype: str
tipo_minerio: <StringArray>
['Minério misto',      'Hematita',     'Magnetita',     'Itabirito',         'Misto',      'HEMATITA',          'Mag.',     'ITABIRITO', 'minerio misto',
     'itabirito', 'MINÉRIO MISTO',         'Itab.',          'Hem.',      'hematita',     'magnetita',     'MAGNETITA']
Length: 16, dtype: str
equipamento_escavacao: <StringArray>
['ESC-05', 'ESC-02', 'ESC-04', 'ESC-01', 'ESC-03']
Length: 5, dtype: str
equipamento_transporte: <StringArray>
['CAM-102', 'CAM-103', 'CAM

**Interpretação do resultado.** Há 150 linhas duplicadas integralmente (inclusive o `registro_id`). Todas as colunas foram lidas como texto; entre as numéricas, `custo_operacional_rs` mistura o formato `20076.99` com `R$ 30.037,21`. As colunas `turno`, `tipo_minerio` e `status_operacao` possuem várias grafias para o mesmo conceito (caixa, acentos, espaços e abreviações como `Hem.` e `Itab.`). Os campos vazios aparecem em teores, umidade, recuperação, consumos, custo de manutenção, índice de qualidade, tipo de minério e equipamentos.

## 3. Padronização estrutural e remoção de duplicidades

Os nomes das colunas já estão em `snake_case`; a função de padronização é aplicada mesmo assim para garantir o padrão. Espaços nas extremidades dos textos são removidos e, em seguida, as duplicidades integrais.

In [4]:
def padronizar_nome_coluna(nome):
    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode("ascii")
    nome = nome.lower().strip()
    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    return nome.strip("_")


df_limpo = df_bruto.copy()
df_limpo.columns = [padronizar_nome_coluna(coluna) for coluna in df_limpo.columns]

for coluna in df_limpo.columns:
    df_limpo[coluna] = df_limpo[coluna].map(
        lambda valor: valor.strip() if isinstance(valor, str) else valor
    )

linhas_antes = len(df_limpo)
df_limpo = df_limpo.drop_duplicates().copy()
print(f"Duplicidades removidas: {linhas_antes - len(df_limpo)}")
print(f"Linhas restantes: {len(df_limpo)}")
assert df_limpo["registro_id"].is_unique

Duplicidades removidas: 150
Linhas restantes: 9850


**Interpretação do resultado.** São removidas 150 linhas e a base volta a ter 9.850 registros com `registro_id` único.

## 4. Conversão de tipos numéricos e temporais

Valores monetários no formato brasileiro (`R$ 30.037,21`) têm o símbolo e o separador de milhar removidos, e a vírgula decimal trocada por ponto. Textos que não podem ser convertidos viram `NaN` (`errors="coerce"`).

A data deveria estar em `dd/mm/aaaa`, mas parte dos registros está em `mm/dd/aaaa` (ex.: `07/30/2025`). A conversão tenta primeiro o formato brasileiro; quando falha ou gera data posterior ao período da base, tenta o formato americano.

In [5]:
def converter_numero_br(valor):
    if pd.isna(valor):
        return np.nan
    texto = str(valor).strip().replace("R$", "").replace(" ", "")
    if "," in texto:
        texto = texto.replace(".", "").replace(",", ".")
    return pd.to_numeric(texto, errors="coerce")


colunas_texto = ["registro_id", "data_operacao", "mina", "estado", "frente_lavra", "turno",
                 "tipo_minerio", "equipamento_escavacao", "equipamento_transporte",
                 "operador_id", "status_operacao"]
conversoes_numericas = [c for c in df_limpo.columns if c not in colunas_texto]

for coluna in conversoes_numericas:
    df_limpo[coluna] = df_limpo[coluna].map(converter_numero_br)

data_br = pd.to_datetime(df_limpo["data_operacao"], format="%d/%m/%Y", errors="coerce")
data_us = pd.to_datetime(df_limpo["data_operacao"], format="%m/%d/%Y", errors="coerce")
usar_us = data_br.isna() | (data_br > DATA_REFERENCIA)
print(f"Datas corrigidas a partir do formato mm/dd/aaaa: {int((usar_us & data_us.notna()).sum())}")
df_limpo["data_operacao"] = data_br.where(~usar_us, data_us)

display(df_limpo[conversoes_numericas + ["data_operacao"]].isna().sum().to_frame("ausentes_apos_conversao"))

Datas corrigidas a partir do formato mm/dd/aaaa: 48


,ausentes_apos_conversao
teor_fe_pct,93
teor_si_pct,113
umidade_minerio_pct,130
granulometria_media_mm,0
massa_extraida_t,0
massa_processada_t,0
massa_produto_t,0
recuperacao_metalurgica_pct,105
rejeito_gerado_t,0
distancia_transporte_km,0


**Interpretação do resultado.** Todas as colunas de medida passam a ser numéricas, incluindo os 149 custos operacionais que estavam como `R$`. As 48 datas em formato americano são recuperadas sem perda. As ausências que aparecem já existiam na base como campos vazios.

## 5. Validação de domínio

Limites definidos pelo conhecimento do processo de mineração, não pela distribuição observada:

| Coluna | Regra | Justificativa |
|---|---|---|
| teor_fe_pct | 0 a 72 | o mineral de maior teor (magnetita) tem ~72,4% de Fe |
| teor_si_pct, umidade_minerio_pct | 0 a 100 / 0 a 30 | percentual; umidade acima de 30% não é minério lavrável |
| recuperacao_metalurgica_pct | 0 a 100 | não se recupera mais do que foi processado |
| indice_qualidade_minerio | 0 a 100 | escala do índice |
| distância, consumos, horas, custos | ≥ 0 | grandezas físicas/monetárias não negativas |
| massas | produto ≤ processada ≤ extraída | balanço de massa |

Valores que violam essas regras são transformados em ausência. Outliers legítimos (valores raros, porém possíveis) são tratados depois, com z-score.

In [6]:
regras_validade = {
    "teor_fe_pct": df_limpo["teor_fe_pct"].between(0, 72),
    "teor_si_pct": df_limpo["teor_si_pct"].between(0, 100),
    "umidade_minerio_pct": df_limpo["umidade_minerio_pct"].between(0, 30),
    "recuperacao_metalurgica_pct": df_limpo["recuperacao_metalurgica_pct"].between(0, 100),
    "indice_qualidade_minerio": df_limpo["indice_qualidade_minerio"].between(0, 100),
    "granulometria_media_mm": df_limpo["granulometria_media_mm"] > 0,
    "massa_extraida_t": df_limpo["massa_extraida_t"] > 0,
    "massa_processada_t": df_limpo["massa_processada_t"].between(0, df_limpo["massa_extraida_t"]),
    "massa_produto_t": df_limpo["massa_produto_t"].between(0, df_limpo["massa_processada_t"]),
    "rejeito_gerado_t": df_limpo["rejeito_gerado_t"] >= 0,
    "distancia_transporte_km": df_limpo["distancia_transporte_km"] >= 0,
    "tempo_ciclo_min": df_limpo["tempo_ciclo_min"] > 0,
    "qtd_viagens": df_limpo["qtd_viagens"] >= 0,
    "consumo_diesel_l": df_limpo["consumo_diesel_l"] >= 0,
    "consumo_diesel_estimado_l": df_limpo["consumo_diesel_estimado_l"] >= 0,
    "consumo_energia_kwh": df_limpo["consumo_energia_kwh"] >= 0,
    "horas_operacao": df_limpo["horas_operacao"].between(0, 24),
    "horas_parada": df_limpo["horas_parada"].between(0, 24),
    "qtd_falhas_equipamento": df_limpo["qtd_falhas_equipamento"] >= 0,
    "horas_desde_manutencao": df_limpo["horas_desde_manutencao"] >= 0,
    "custo_manutencao_rs": df_limpo["custo_manutencao_rs"] >= 0,
    "custo_operacional_rs": df_limpo["custo_operacional_rs"] >= 0,
    "custo_operacional_estimado_rs": df_limpo["custo_operacional_estimado_rs"] >= 0,
    "produtividade_t_h": df_limpo["produtividade_t_h"] >= 0,
    "emissao_co2_kg": df_limpo["emissao_co2_kg"] >= 0,
}

relatorio_invalidos = []
for coluna, mascara_valida in regras_validade.items():
    mascara_invalida = df_limpo[coluna].notna() & ~mascara_valida
    relatorio_invalidos.append({"coluna": coluna, "valores_invalidos": int(mascara_invalida.sum())})
    df_limpo.loc[mascara_invalida, coluna] = np.nan

relatorio_invalidos = pd.DataFrame(relatorio_invalidos)
display(relatorio_invalidos[relatorio_invalidos["valores_invalidos"] > 0])

,coluna,valores_invalidos
0,teor_fe_pct,14
2,umidade_minerio_pct,11
3,recuperacao_metalurgica_pct,12
4,indice_qualidade_minerio,7
10,distancia_transporte_km,10
13,consumo_diesel_l,10
19,horas_desde_manutencao,9


### Recuperação metalúrgica recalculada

Na base, `recuperacao_metalurgica_pct` é exatamente `massa_produto_t / massa_processada_t × 100`. Quando o valor está ausente ou foi invalidado, ele pode ser **recalculado** a partir das massas, o que é mais correto do que imputar uma mediana.

In [7]:
recuperacao_calculada = (df_limpo["massa_produto_t"] / df_limpo["massa_processada_t"] * 100).round(2)
faltando = df_limpo["recuperacao_metalurgica_pct"].isna() & recuperacao_calculada.notna()
df_limpo.loc[faltando, "recuperacao_metalurgica_pct"] = recuperacao_calculada[faltando]
print(f"Recuperações recalculadas pelas massas: {int(faltando.sum())}")

Recuperações recalculadas pelas massas: 117


**Interpretação do resultado.** Teores de Fe acima de 72%, umidades acima de 100%, índices de qualidade acima de 100, distâncias, consumos de diesel e horas desde manutenção negativos são convertidos em ausência. As recuperações ausentes ou acima de 100% são reconstruídas pelas massas, sem nenhum `NaN` restante nessa coluna.

## 6. Normalização categórica e binarização

Cada rótulo é convertido para uma forma auxiliar (sem acento, minúsculo, sem espaços extras) e mapeado para o rótulo canônico por um dicionário explícito. Rótulos não reconhecidos e campos vazios viram ausência.

A variável-alvo `operacao_com_restricao` é criada a partir de `status_operacao`.

In [8]:
def normalizar_rotulo(valor):
    if pd.isna(valor):
        return None
    texto = unicodedata.normalize("NFKD", str(valor)).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", texto.strip().lower())


def mapear_categoria(serie, mapa):
    return serie.map(lambda valor: mapa.get(normalizar_rotulo(valor), np.nan))


def mapa_identidade(valores):
    return {normalizar_rotulo(v): v for v in valores}


mapas_categorias = {
    "mina": mapa_identidade(["Mina Azul", "Mina Brucutu", "Mina Catalão", "Mina Corumbá",
                             "Mina Itabira", "Mina Serra Norte"]),
    "estado": mapa_identidade(["GO", "MG", "MS", "PA"]),
    "frente_lavra": mapa_identidade([f"F0{i}" for i in range(1, 7)]),
    "turno": {"manha": "Manhã", "tarde": "Tarde", "noite": "Noite"},
    "tipo_minerio": {
        "hematita": "Hematita", "hem.": "Hematita",
        "itabirito": "Itabirito", "itab.": "Itabirito",
        "magnetita": "Magnetita", "mag.": "Magnetita",
        "minerio misto": "Minério misto", "misto": "Minério misto",
    },
    "equipamento_escavacao": mapa_identidade([f"ESC-0{i}" for i in range(1, 6)]),
    "equipamento_transporte": mapa_identidade([f"CAM-10{i}" for i in range(1, 9)]),
    "status_operacao": {
        "operacao normal": "Operação normal", "normal": "Operação normal",
        "operacao com restricao": "Operação com restrição", "restricao": "Operação com restrição",
        "operacao critica": "Operação crítica",
    },
}

for coluna, mapa in mapas_categorias.items():
    df_limpo[coluna] = mapear_categoria(df_limpo[coluna], mapa)

df_limpo["operacao_com_restricao"] = df_limpo["status_operacao"].map(
    {"Operação normal": 0, "Operação com restrição": 1, "Operação crítica": 1}
)

for coluna in mapas_categorias:
    print(f"{coluna}: {df_limpo[coluna].value_counts(dropna=False).to_dict()}")

mina: {'Mina Brucutu': 1670, 'Mina Azul': 1667, 'Mina Corumbá': 1649, 'Mina Serra Norte': 1642, 'Mina Catalão': 1619, 'Mina Itabira': 1603}


estado: {'PA': 3309, 'MG': 3273, 'MS': 1649, 'GO': 1619}
frente_lavra: {'F01': 1668, 'F06': 1658, 'F02': 1650, 'F04': 1646, 'F03': 1622, 'F05': 1606}
turno: {'Manhã': 3729, 'Tarde': 3504, 'Noite': 2617}
tipo_minerio: {'Hematita': 3741, 'Itabirito': 3095, 'Magnetita': 1547, 'Minério misto': 1349, nan: 118}
equipamento_escavacao: {'ESC-05': 2010, 'ESC-01': 1975, 'ESC-04': 1933, 'ESC-03': 1929, 'ESC-02': 1903, nan: 100}
equipamento_transporte: {'CAM-105': 1267, 'CAM-108': 1253, 'CAM-107': 1244, 'CAM-101': 1233, 'CAM-104': 1231, 'CAM-106': 1210, 'CAM-103': 1162, 'CAM-102': 1141, nan: 109}
status_operacao: {'Operação normal': 7186, 'Operação com restrição': 2650, 'Operação crítica': 14}


**Interpretação do resultado.** `turno` passa de 12 para 3 níveis, `tipo_minerio` de 17 para 4 (mais ausências) e `status_operacao` de 9 para 3. A relação mina → estado é 1 para 1 (ex.: Mina Azul e Serra Norte → PA), portanto `estado` é redundante para modelagem e será mantido apenas na base limpa.

## 7. Tratamento do alvo e engenharia de atributos temporais

Registros sem alvo válido seriam removidos (o alvo nunca é imputado). A data é transformada em atributos numéricos: ano, mês, dia da semana e dias desde o início do período.

In [9]:
linhas_sem_alvo = int(df_limpo["operacao_com_restricao"].isna().sum())
df_limpo = df_limpo.dropna(subset=["operacao_com_restricao"]).copy()
df_limpo["operacao_com_restricao"] = df_limpo["operacao_com_restricao"].astype(int)

inicio_periodo = df_limpo["data_operacao"].min()
df_limpo["ano_operacao"] = df_limpo["data_operacao"].dt.year
df_limpo["mes_operacao"] = df_limpo["data_operacao"].dt.month
df_limpo["dia_semana_operacao"] = df_limpo["data_operacao"].dt.dayofweek
df_limpo["dias_desde_inicio"] = (df_limpo["data_operacao"] - inicio_periodo).dt.days

inteiras = ["qtd_viagens", "qtd_falhas_equipamento"]
df_limpo[inteiras] = df_limpo[inteiras].astype("Int64")

print(f"Linhas removidas por ausência do alvo: {linhas_sem_alvo}")
print(f"Dimensão da base limpa: {df_limpo.shape}")
display(df_limpo.head())
display(df_limpo.isna().sum()[lambda s: s > 0].sort_values(ascending=False).to_frame("ausentes"))

Linhas removidas por ausência do alvo: 0
Dimensão da base limpa: (9850, 41)


,registro_id,data_operacao,mina,estado,frente_lavra,turno,tipo_minerio,equipamento_escavacao,equipamento_transporte,operador_id,teor_fe_pct,teor_si_pct,umidade_minerio_pct,granulometria_media_mm,massa_extraida_t,massa_processada_t,massa_produto_t,recuperacao_metalurgica_pct,rejeito_gerado_t,distancia_transporte_km,tempo_ciclo_min,qtd_viagens,consumo_diesel_l,consumo_diesel_estimado_l,consumo_energia_kwh,horas_operacao,horas_parada,qtd_falhas_equipamento,horas_desde_manutencao,custo_manutencao_rs,custo_operacional_rs,custo_operacional_estimado_rs,produtividade_t_h,emissao_co2_kg,indice_qualidade_minerio,status_operacao,operacao_com_restricao,ano_operacao,mes_operacao,dia_semana_operacao,dias_desde_inicio
0,REG-0000001,2026-02-16,Mina Brucutu,MG,F02,Manhã,Minério misto,ESC-05,CAM-102,OP-1713,51.900,6.640,12.580,24.100,"1,544.550","1,513.060","1,271.050",84.010,242.010,4.830,34.300,7,90.640,85.340,340.810,6.390,2.490,1,51.400,"12,703.400","20,076.990","19,935.590",174.020,243.880,76.400,Operação normal,0,2026,2,0,411
1,REG-0000002,2026-02-06,Mina Catalão,GO,F01,Manhã,Hematita,ESC-02,CAM-103,OP-1425,67.090,1.000,6.880,43.400,"1,517.530","1,460.850","1,338.770",91.640,122.080,7.070,34.000,6,84.480,84.980,208.410,9.110,1.340,0,114.000,"2,252.470","8,828.680","9,233.370",145.270,227.440,100.000,Operação normal,0,2026,2,4,401
2,REG-0000003,2026-06-06,Mina Catalão,GO,F06,Tarde,Magnetita,ESC-04,CAM-102,OP-2410,55.990,8.300,4.960,23.100,"1,627.570","1,554.400","1,388.760",89.340,165.640,3.600,30.700,8,68.970,64.570,477.480,9.880,3.350,1,26.200,"22,251.160","33,266.530","32,896.490",122.970,187.480,86.600,Operação normal,0,2026,6,5,521
3,REG-0000004,2026-04-09,Mina Catalão,GO,F04,Manhã,Itabirito,ESC-05,CAM-106,OP-1754,55.570,15.440,8.120,61.400,"2,680.670","2,526.330","2,013.640",79.710,512.700,4.920,39.000,15,141.940,130.190,555.290,13.740,1.500,1,123.000,"22,495.550","33,019.780","32,892.620",175.870,382.510,78.500,Operação normal,0,2026,4,3,463
4,REG-0000005,2025-02-07,Mina Corumbá,MS,F03,Manhã,Itabirito,ESC-05,CAM-105,OP-1726,46.910,18.450,4.060,59.800,"1,596.040","1,540.210","1,285.540",83.470,254.670,5.420,37.000,7,74.810,74.890,497.120,7.000,2.140,1,91.800,"23,488.640","29,482.430","29,417.870",174.660,199.510,62.300,Operação normal,0,2025,2,4,37


,ausentes
indice_qualidade_minerio,157
umidade_minerio_pct,141
consumo_energia_kwh,137
consumo_diesel_l,132
tipo_minerio,118
custo_manutencao_rs,115
teor_si_pct,113
equipamento_transporte,109
teor_fe_pct,107
equipamento_escavacao,100


**Interpretação do resultado.** Nenhuma linha perde o alvo. A base limpa tem 9.850 operações; as ausências restantes estão em variáveis explicativas e serão imputadas somente na matriz de ML, com estatísticas do treino.

### Exportação da base limpa

Esta é a tabela que será usada na próxima aula para análise e dashboard: legível, tipada e sem duplicidades. Datas saem no formato ISO (`aaaa-mm-dd`).

In [10]:
df_limpo.to_csv("base_14_empresa_mineracao_limpa.csv", index=False, sep=";", encoding="utf-8-sig")
print("Base limpa exportada: base_14_empresa_mineracao_limpa.csv")

Base limpa exportada: base_14_empresa_mineracao_limpa.csv


## 8. Separação entre treino e teste

São excluídos da matriz: identificadores (`registro_id`, `operador_id` — 1.497 valores, sem significado físico), a data original, `estado` (redundante com `mina`) e `status_operacao` (origem do alvo).

In [11]:
X = df_limpo.drop(columns=["operacao_com_restricao", "status_operacao", "registro_id",
                            "operador_id", "data_operacao", "estado"])
y = df_limpo["operacao_com_restricao"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEMENTE, stratify=y
)

print(f"Treino: {X_train.shape[0]} linhas | Teste: {X_test.shape[0]} linhas")
display(pd.DataFrame({
    "base_completa": y.value_counts(normalize=True).sort_index(),
    "treino": y_train.value_counts(normalize=True).sort_index(),
    "teste": y_test.value_counts(normalize=True).sort_index(),
}).round(3))

Treino: 7880 linhas | Teste: 1970 linhas


,base_completa,treino,teste
operacao_com_restricao,,,
0,0.730,0.730,0.729
1,0.270,0.270,0.271


## 9. Outliers e z-score

Limites de ±3 desvios-padrão aprendidos no treino e aplicados por *clipping* ao treino e ao teste. Contagens e atributos de data não são limitados.

In [12]:
colunas_categoricas = ["mina", "frente_lavra", "turno", "tipo_minerio",
                       "equipamento_escavacao", "equipamento_transporte"]
colunas_numericas = [c for c in X.columns if c not in colunas_categoricas]
colunas_outliers = [c for c in colunas_numericas if c not in
                    ["qtd_viagens", "qtd_falhas_equipamento", "ano_operacao", "mes_operacao",
                     "dia_semana_operacao", "dias_desde_inicio"]]


def aprender_limites_z(df, colunas, limite_z=3.0):
    limites, relatorio = {}, []
    for coluna in colunas:
        media = df[coluna].mean(skipna=True)
        desvio = df[coluna].std(skipna=True, ddof=0)
        inferior, superior = media - limite_z * desvio, media + limite_z * desvio
        limites[coluna] = (inferior, superior)
        relatorio.append({
            "coluna": coluna, "media_treino": media, "desvio_treino": desvio,
            "limite_inferior": inferior, "limite_superior": superior,
            "outliers_treino": int(((df[coluna] < inferior) | (df[coluna] > superior)).sum()),
        })
    return limites, pd.DataFrame(relatorio)


def aplicar_clipping(df, limites):
    resultado = df.copy()
    for coluna, (inferior, superior) in limites.items():
        resultado[coluna] = resultado[coluna].clip(lower=inferior, upper=superior)
    return resultado


limites_z, relatorio_z = aprender_limites_z(X_train, colunas_outliers)
X_train_tratado = aplicar_clipping(X_train, limites_z)
X_test_tratado = aplicar_clipping(X_test, limites_z)
display(relatorio_z.round(2))

,coluna,media_treino,desvio_treino,limite_inferior,limite_superior,outliers_treino
0,teor_fe_pct,56.410,8.070,32.190,80.620,0
1,teor_si_pct,10.310,5.970,-7.590,28.210,0
2,umidade_minerio_pct,9.520,3.300,-0.370,19.420,9
3,granulometria_media_mm,34.020,11.960,-1.860,69.900,11
4,massa_extraida_t,"1,848.520",524.750,274.260,"3,422.780",18
5,massa_processada_t,"1,774.830",504.540,261.220,"3,288.430",19
6,massa_produto_t,"1,585.690",463.260,195.900,"2,975.490",25
7,recuperacao_metalurgica_pct,89.340,5.600,72.550,106.130,5
8,rejeito_gerado_t,189.140,116.070,-159.070,537.340,48
9,distancia_transporte_km,6.220,2.330,-0.770,13.210,15


## 10. Imputação de valores ausentes

Mediana para numéricas e moda para categóricas, ajustadas somente no treino.

In [13]:
X_train_tratado[colunas_numericas] = X_train_tratado[colunas_numericas].astype(float)
X_test_tratado[colunas_numericas] = X_test_tratado[colunas_numericas].astype(float)
X_train_tratado[colunas_categoricas] = X_train_tratado[colunas_categoricas].astype(object)
X_test_tratado[colunas_categoricas] = X_test_tratado[colunas_categoricas].astype(object)

imputador_numerico = SimpleImputer(strategy="median")
imputador_categorico = SimpleImputer(strategy="most_frequent")

X_train_imputado = X_train_tratado.copy()
X_test_imputado = X_test_tratado.copy()
X_train_imputado[colunas_numericas] = imputador_numerico.fit_transform(X_train_tratado[colunas_numericas])
X_test_imputado[colunas_numericas] = imputador_numerico.transform(X_test_tratado[colunas_numericas])
X_train_imputado[colunas_categoricas] = imputador_categorico.fit_transform(X_train_tratado[colunas_categoricas])
X_test_imputado[colunas_categoricas] = imputador_categorico.transform(X_test_tratado[colunas_categoricas])

print(f"Ausências no treino após imputação: {int(X_train_imputado.isna().sum().sum())}")
print(f"Ausências no teste após imputação: {int(X_test_imputado.isna().sum().sum())}")

Ausências no treino após imputação: 0
Ausências no teste após imputação: 0


## 11. One-hot encoding

In [14]:
codificador = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float64)
treino_categorias = codificador.fit_transform(X_train_imputado[colunas_categoricas])
teste_categorias = codificador.transform(X_test_imputado[colunas_categoricas])
nomes_categorias = codificador.get_feature_names_out(colunas_categoricas)

X_train_codificado = pd.concat([
    X_train_imputado.drop(columns=colunas_categoricas),
    pd.DataFrame(treino_categorias, columns=nomes_categorias, index=X_train_imputado.index),
], axis=1)
X_test_codificado = pd.concat([
    X_test_imputado.drop(columns=colunas_categoricas),
    pd.DataFrame(teste_categorias, columns=nomes_categorias, index=X_test_imputado.index),
], axis=1)
print(f"Variáveis antes: {X_train_imputado.shape[1]} | depois: {X_train_codificado.shape[1]}")

Variáveis antes: 35 | depois: 61


## 12. Padronização z-score das variáveis numéricas

In [15]:
escalonador = StandardScaler()
X_train_pronto = X_train_codificado.copy()
X_test_pronto = X_test_codificado.copy()
X_train_pronto[colunas_numericas] = escalonador.fit_transform(X_train_codificado[colunas_numericas])
X_test_pronto[colunas_numericas] = escalonador.transform(X_test_codificado[colunas_numericas])
display(X_train_pronto[colunas_numericas].agg(["mean", "std"]).T.round(3))

,mean,std
teor_fe_pct,0.000,1.000
teor_si_pct,-0.000,1.000
umidade_minerio_pct,0.000,1.000
granulometria_media_mm,0.000,1.000
massa_extraida_t,-0.000,1.000
massa_processada_t,0.000,1.000
massa_produto_t,-0.000,1.000
recuperacao_metalurgica_pct,0.000,1.000
rejeito_gerado_t,0.000,1.000
distancia_transporte_km,0.000,1.000


## 13. Validação da matriz final

In [16]:
assert X_train_pronto.isna().sum().sum() == 0
assert X_test_pronto.isna().sum().sum() == 0
assert X_train_pronto.columns.equals(X_test_pronto.columns)
assert all(pd.api.types.is_numeric_dtype(tipo) for tipo in X_train_pronto.dtypes)
assert set(y_train.unique()).issubset({0, 1}) and set(y_test.unique()).issubset({0, 1})
assert set(np.unique(X_train_pronto[nomes_categorias].to_numpy())).issubset({0.0, 1.0})
print("Validação concluída: as matrizes estão prontas para machine learning.")

Validação concluída: as matrizes estão prontas para machine learning.


## 14. Exportação dos resultados

In [17]:
treino_final = X_train_pronto.copy()
treino_final["operacao_com_restricao"] = y_train.loc[treino_final.index].astype(int)
teste_final = X_test_pronto.copy()
teste_final["operacao_com_restricao"] = y_test.loc[teste_final.index].astype(int)

with pd.ExcelWriter("resultado_data_wrangling_mineracao.xlsx", engine="openpyxl") as escritor:
    df_bruto.to_excel(escritor, sheet_name="Dados_brutos", index=False)
    df_limpo.to_excel(escritor, sheet_name="Dados_limpos", index=False)
    treino_final.to_excel(escritor, sheet_name="Treino_pronto_ML", index=False)
    teste_final.to_excel(escritor, sheet_name="Teste_pronto_ML", index=False)

treino_final.to_csv("dados_prontos_ml_treino.csv", index=False, encoding="utf-8-sig")
teste_final.to_csv("dados_prontos_ml_teste.csv", index=False, encoding="utf-8-sig")
print("Arquivos gerados: resultado_data_wrangling_mineracao.xlsx, base_14_empresa_mineracao_limpa.csv, "
      "dados_prontos_ml_treino.csv, dados_prontos_ml_teste.csv")

Arquivos gerados: resultado_data_wrangling_mineracao.xlsx, base_14_empresa_mineracao_limpa.csv, dados_prontos_ml_treino.csv, dados_prontos_ml_teste.csv
